In [1]:
"""
diagnose_display.py
===================
Run this script on the Linux machine to diagnose why live_plots popups
don't appear.  Each section is independent — run the whole thing and
share the output.

    python diagnose_display.py
"""

import sys
import os
import subprocess

SEP = "-" * 60

def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print('='*60)

# ─────────────────────────────────────────────────────────────────────────────
# 1. Display environment
# ─────────────────────────────────────────────────────────────────────────────
section("1. Display environment variables")

for var in ("DISPLAY", "WAYLAND_DISPLAY", "XDG_SESSION_TYPE",
            "XDG_RUNTIME_DIR", "MPLBACKEND"):
    val = os.environ.get(var, "<not set>")
    print(f"  {var:25s} = {val}")

# ─────────────────────────────────────────────────────────────────────────────
# 2. Python / platform
# ─────────────────────────────────────────────────────────────────────────────
section("2. Python / platform")
print(f"  Python        : {sys.version}")
print(f"  Executable    : {sys.executable}")
import platform
print(f"  Platform      : {platform.platform()}")
print(f"  Desktop env   : {os.environ.get('XDG_CURRENT_DESKTOP', '<not set>')}")
print(f"  SSH_CLIENT    : {os.environ.get('SSH_CLIENT', '<not set>')}")
print(f"  SSH_TTY       : {os.environ.get('SSH_TTY', '<not set>')}")

# ─────────────────────────────────────────────────────────────────────────────
# 3. GUI toolkit availability
# ─────────────────────────────────────────────────────────────────────────────
section("3. GUI toolkit imports")

toolkits = {
    "tkinter"  : "TkAgg  backend",
    "PyQt5"    : "Qt5Agg backend",
    "PyQt6"    : "Qt6Agg backend",
    "PySide2"  : "Qt5 (PySide2)",
    "PySide6"  : "Qt6 (PySide6)",
    "wx"       : "wxAgg  backend",
}
available = []
for mod, label in toolkits.items():
    try:
        __import__(mod)
        print(f"  [OK]   {mod:12s}  ({label})")
        available.append(mod)
    except ImportError as e:
        print(f"  [MISS] {mod:12s}  ({label})  — {e}")

if not available:
    print("\n  *** NO GUI toolkit found — all backends will fall back to Agg ***")

# ─────────────────────────────────────────────────────────────────────────────
# 4. Matplotlib backend resolution
# ─────────────────────────────────────────────────────────────────────────────
section("4. Matplotlib backend resolution")

try:
    import matplotlib
    print(f"  matplotlib version : {matplotlib.__version__}")
    print(f"  config dir         : {matplotlib.get_configdir()}")
    print(f"  MPLBACKEND env     : {os.environ.get('MPLBACKEND', '<not set>')}")

    # What _pick_backend() would choose
    import importlib
    chosen = "Agg (fallback — no GUI toolkit found)"
    for backend in ("TkAgg", "Qt5Agg", "Qt6Agg", "wxAgg", "MacOSX"):
        mod = {"TkAgg": "tkinter", "Qt5Agg": "PyQt5", "Qt6Agg": "PyQt6",
               "wxAgg": "wx", "MacOSX": "AppKit"}.get(backend, backend)
        try:
            importlib.import_module(mod)
            chosen = backend
            break
        except ImportError:
            continue
    print(f"  _pick_backend() would choose : {chosen}")

    # What matplotlib itself would auto-select
    matplotlib.use("Agg")   # safe import; don't actually open a window here
    import matplotlib.pyplot as plt
    print(f"  Current backend after import : {matplotlib.get_backend()}")

except Exception as e:
    print(f"  ERROR: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# 5. Can a subprocess open a window?
# ─────────────────────────────────────────────────────────────────────────────
section("5. Subprocess window test (non-blocking, 3-second window)")

test_script = """
import os, sys, time

# Pass through display env explicitly (may not be inherited on some systems)
import matplotlib
backend_order = ["TkAgg","Qt5Agg","Qt6Agg","wxAgg","MacOSX"]
chosen = "Agg"
import importlib
for b in backend_order:
    mod = {"TkAgg":"tkinter","Qt5Agg":"PyQt5","Qt6Agg":"PyQt6",
           "wxAgg":"wx","MacOSX":"AppKit"}.get(b, b)
    try:
        importlib.import_module(mod)
        chosen = b
        break
    except ImportError:
        pass

print(f"subprocess backend: {chosen}", flush=True)
matplotlib.use(chosen)
import matplotlib.pyplot as plt

if chosen == "Agg":
    print("Agg selected — cannot open a window", flush=True)
    sys.exit(1)

try:
    fig, ax = plt.subplots()
    ax.set_title("Test window — should close in 3 s")
    ax.plot([1,2,3],[1,4,9])
    plt.tight_layout()
    # Non-blocking show + pause
    plt.show(block=False)
    plt.pause(3)
    plt.close(fig)
    print("Window opened and closed OK", flush=True)
    sys.exit(0)
except Exception as e:
    print(f"Window open FAILED: {e}", flush=True)
    sys.exit(2)
"""

try:
    # Forward display environment to subprocess
    env = os.environ.copy()
    result = subprocess.run(
        [sys.executable, "-c", test_script],
        capture_output=True, text=True, timeout=15, env=env,
    )
    print(f"  exit code : {result.returncode}")
    if result.stdout:
        for line in result.stdout.strip().splitlines():
            print(f"  stdout: {line}")
    if result.stderr:
        for line in result.stderr.strip().splitlines()[:20]:
            print(f"  stderr: {line}")
except subprocess.TimeoutExpired:
    print("  TIMEOUT — subprocess hung (window may have opened but plt.pause blocked)")
except Exception as e:
    print(f"  ERROR running subprocess test: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. multiprocessing spawn test
# ─────────────────────────────────────────────────────────────────────────────
section("6. multiprocessing spawn context test")

import multiprocessing as mp

def _spawn_test(q):
    import os
    try:
        import matplotlib
        import importlib
        chosen = "Agg"
        for b in ("TkAgg","Qt5Agg","Qt6Agg","wxAgg","MacOSX"):
            mod = {"TkAgg":"tkinter","Qt5Agg":"PyQt5","Qt6Agg":"PyQt6",
                   "wxAgg":"wx","MacOSX":"AppKit"}.get(b,b)
            try:
                importlib.import_module(mod)
                chosen = b; break
            except ImportError:
                pass
        matplotlib.use(chosen)
        import matplotlib.pyplot as plt
        q.put({"backend": chosen,
               "DISPLAY": os.environ.get("DISPLAY","<not set>"),
               "WAYLAND_DISPLAY": os.environ.get("WAYLAND_DISPLAY","<not set>"),
               "error": None})
    except Exception as e:
        q.put({"backend": "ERROR", "error": str(e)})

try:
    ctx = mp.get_context("spawn")
    q = ctx.Queue()
    p = ctx.Process(target=_spawn_test, args=(q,), daemon=True)
    p.start()
    p.join(timeout=10)
    if not q.empty():
        result = q.get()
        print(f"  backend in spawn : {result.get('backend')}")
        print(f"  DISPLAY in spawn : {result.get('DISPLAY')}")
        print(f"  WAYLAND in spawn : {result.get('WAYLAND_DISPLAY')}")
        if result.get("error"):
            print(f"  error            : {result['error']}")
    else:
        print("  spawn process produced no output (timeout or crash)")
except Exception as e:
    print(f"  ERROR: {e}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. Summary and recommendations
# ─────────────────────────────────────────────────────────────────────────────
section("7. Summary and likely fixes")

display_set  = bool(os.environ.get("DISPLAY") or os.environ.get("WAYLAND_DISPLAY"))
is_ssh       = bool(os.environ.get("SSH_CLIENT") or os.environ.get("SSH_TTY"))
has_gui_tk   = "tkinter" in available
has_gui_qt   = any(m in available for m in ("PyQt5","PyQt6","PySide2","PySide6"))
has_any_gui  = bool(available)

if not display_set:
    print("""
  PROBLEM: No DISPLAY or WAYLAND_DISPLAY variable set.
  The subprocess has no display server to draw on.

  Fixes:
    A) If using SSH — reconnect with X forwarding:
         ssh -X user@host       (basic X11 forwarding)
         ssh -Y user@host       (trusted, faster)
       Then check:  echo $DISPLAY  (should be e.g. localhost:10.0)

    B) If running on a local Linux desktop but inside a terminal that
       strips the env, export it manually before launching Jupyter:
         export DISPLAY=:0
         jupyter notebook

    C) If the machine is truly headless (no monitor), use VNC or
       Xvfb (virtual framebuffer):
         Xvfb :99 -screen 0 1024x768x24 &
         export DISPLAY=:99
""")
elif is_ssh:
    print("""
  PROBLEM: Running over SSH. DISPLAY is set but X forwarding may be
  slow or blocked by the server's sshd config.

  Fixes:
    • Confirm X forwarding is enabled on the server:
        grep -i "X11Forwarding" /etc/ssh/sshd_config   # should be "yes"
    • Use -Y (trusted) instead of -X if windows don't appear:
        ssh -Y user@host
    • For better performance consider using VNC instead of X forwarding.
""")
elif not has_any_gui:
    print("""
  PROBLEM: DISPLAY is set but no GUI toolkit is installed in this
  Python environment, so matplotlib falls back to Agg (no window).

  Fixes (install at least one):
    pip install pyqt5          # easiest, usually works out of the box
    conda install pyqt         # if using conda
    sudo apt install python3-tk  # for Tkinter (system-level)
""")
elif not has_gui_qt and has_gui_tk:
    print("""
  STATUS: tkinter found — TkAgg backend will be used.
  If windows still don't appear, check section 5/6 output above for
  runtime errors (often a missing libtk or display permission issue).
""")
else:
    print("""
  Environment looks OK. If windows still don't appear, the issue is
  likely a runtime error inside the subprocess. Check section 5 and 6
  output above for specific error messages.
""")

print(SEP)
print("Diagnostics complete.")
print(SEP)


  1. Display environment variables
  DISPLAY                   = :0
  WAYLAND_DISPLAY           = wayland-0
  XDG_SESSION_TYPE          = wayland
  XDG_RUNTIME_DIR           = /run/user/11088
  MPLBACKEND                = module://matplotlib_inline.backend_inline

  2. Python / platform
  Python        : 3.11.2 (main, Apr 28 2025, 14:11:48) [GCC 12.2.0]
  Executable    : /usr/bin/python3
  Platform      : Linux-6.1.0-38-amd64-x86_64-with-glibc2.36
  Desktop env   : GNOME
  SSH_CLIENT    : <not set>
  SSH_TTY       : <not set>

  3. GUI toolkit imports
  [OK]   tkinter       (TkAgg  backend)
  [OK]   PyQt5         (Qt5Agg backend)
  [MISS] PyQt6         (Qt6Agg backend)  — No module named 'PyQt6'
  [MISS] PySide2       (Qt5 (PySide2))  — No module named 'PySide2'
  [MISS] PySide6       (Qt6 (PySide6))  — No module named 'PySide6'
  [OK]   wx            (wxAgg  backend)

  4. Matplotlib backend resolution
  matplotlib version : 3.6.3
  config dir         : /user/rea3/.config/matplotlib


Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/usr/lib/python3.11/multiprocessing/spawn.py", line 120, in spawn_main
    exitcode = _main(fd, parent_sentinel)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.11/multiprocessing/spawn.py", line 130, in _main
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: Can't get attribute '_spawn_test' on <module '__main__' (built-in)>
